# GAPS — the novel method (GBDT-Augmented Polynomial Spectral search)

**Adrian Ymeri · University of Prishtina · SpOC-3**

THESIS §9 found the residual gap to the leaderboard is a **decode-expressiveness
limit**: every ordering is `argsort(F·x)` over *linear* spectral features, and the
leader's solutions lie off that manifold. **GAPS** attacks that directly, with
GBDT as the central mechanism:

> decode score  =  **Phi(F)·x**  +  **β · g_GBDT(F)**

- `Phi(F)` — a **polynomial** expansion of the spectral features (richer basis),
- `g_GBDT(F)` — a **nonlinear** score map, a gradient-boosted tree trained
  DAgger-style on the elite orderings (the off-manifold direction no linear
  policy can express).

Searched by GPU-scaled CMA-ES, warm-started from your banked best. This notebook
runs the **controlled three-way comparison** that tells us whether the novelty
works and whether GBDT earns its place:

1. linear baseline (`gpu_search`) — your −5,405,118,
2. GAPS **poly-only** (`--gbdt-every 0`),
3. GAPS **poly + GBDT** (the full method).

Use a **T4 GPU** runtime. Run top to bottom.

In [ ]:
!nvidia-smi -L
from numba import cuda; assert cuda.is_available(), "Set Runtime -> T4 GPU"
!pip -q install lightgbm xgboost 2>/dev/null
print("ready")

## 1 · Upload & unpack (use the newest `torso_project.zip` — it contains `gaps_search.py`)

In [ ]:
import os, zipfile, glob
from google.colab import files
%cd /content
up = files.upload()
z = next(k for k in up if k.endswith('.zip'))
!rm -rf torso_project
zipfile.ZipFile(z).extractall('.')
ROOT = os.path.dirname(glob.glob('**/tools/gaps_search.py', recursive=True)[0]).rsplit('/tools',1)[0]
%cd {ROOT}
assert os.path.exists('tools/gaps_search.py'), "old zip — re-upload the newest one"
print("gaps_search.py present ✓")

## 2 · Validation gate (must print ALL CHECKS PASSED, 0 mismatches)

In [ ]:
!python3 tools/validate_gpu.py --problem large-graph --batch 512 --cpu-sample 48

## 3 · GAPS poly-only ablation (no GBDT)
Tests whether the **polynomial** expansion alone escapes the linear manifold.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
!PYTHONWARNINGS=ignore python3 tools/gaps_search.py --problem large-graph --pop 4096 --budget 1800 --gbdt-every 0 --algo gaps_nogbdt --seed 42

## 4 · GAPS full method (poly + GBDT) — the novelty
Same everything, but with the GBDT nonlinear column retrained every 10 gens.
If this beats §3, **that gap is GBDT's contribution in GAPS** — a new, positive,
controlled GBDT result inside the novel method.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
!PYTHONWARNINGS=ignore python3 tools/gaps_search.py --problem large-graph --pop 4096 --budget 1800 --gbdt-every 10 --backend lightgbm --algo gaps --seed 42

## 5 · Read the three-way result
Compare the final **Official score** of:
- linear baseline (−5,405,118, your banked best),
- GAPS poly-only (§3),
- GAPS poly+GBDT (§4).

**Interpretation:**
- poly-only > baseline → the polynomial decode escapes the linear manifold (the §9 finding, confirmed constructively).
- poly+GBDT > poly-only → **GBDT improves results inside the novel method** (controlled, your goal #2).
- any line printing `BEAT THE LEADER` → #1.

Download whichever wins and pool it on your Mac:
```
from google.colab import files
files.download('submissions/large-graph/gaps.json')
```
Then `cp` into submissions/large-graph/ and run `tools/portfolio.py` +
`tools/refine_thresholds.py` for the official verified number. Paste me the three
final scores and I'll write the result into the thesis honestly.

## 7 · Multi-seed ablation (rigor) — turns the n=1 result into mean ± std
Runs GAPS with/without GBDT across 5 seeds, then aggregates with the official
scorer. This is the statistically-grounded version of the +24,766 HV ablation.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
for s in (1,2,3,4,5):
    !PYTHONWARNINGS=ignore python3 tools/gaps_search.py --problem large-graph --pop 4096 --budget 900 --poly-extra 0 --gbdt-every 8 --backend lightgbm --algo gaps_s{s} --seed {s}
    !PYTHONWARNINGS=ignore python3 tools/gaps_search.py --problem large-graph --pop 4096 --budget 400 --poly-extra 0 --gbdt-every 0 --backend lightgbm --algo gaps_nogbdt_s{s} --seed {s}
!python3 tools/gaps_ablation_stats.py --problem large-graph

## 8 · Stronger GAPS — the leaderboard attempt (long shot)
LambdaMART GBDT, bigger population, longer budget. The front-gap diagnostic
(`tools/front_gap_analysis.py`) shows the dense core is already optimal and the
residual gap is a small fill-in-quality gap, so this is a genuine but uncertain
attempt — paste the final score and we re-score officially on the Mac.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
!PYTHONWARNINGS=ignore python3 tools/gaps_search.py --problem large-graph --pop 8192 --budget 5400 --poly-extra 0 --gbdt-every 6 --backend lightgbm --rank --algo gaps_strong --seed 11

## 9 · Diagnostics you can run anytime (CPU, fast)
Where is the gap, and is it closable?

In [ ]:
!python3 tools/front_gap_analysis.py --problem large-graph

## 10 · QNE — reproduce the leaderboard winner (+ our GBDT ablation)
Learned from cuda-torso: per-threshold elites + full polynomial features +
neuroevolution. `qne_search.py` is that algorithm on our evaluator. Run the
winner's method, then our GBDT-augmented version, and compare — that ablation is
the novelty question: *does boosted-tree augmentation improve the state of the art?*

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
# faithful winner reproduction (no GBDT)
!PYTHONWARNINGS=ignore python3 tools/qne_search.py --problem large-graph --batch 1024 --budget 3600 --algo qne

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
# OUR contribution: same method + GBDT-learned feature column
!PYTHONWARNINGS=ignore python3 tools/qne_search.py --problem large-graph --batch 1024 --budget 3600 --gbdt --algo qne_gbdt

Compare the two **Official score** lines. If `qne_gbdt` < `qne`, GBDT improves
the SOTA method — a clean, novel result. Download both and re-score on the Mac
(`tools/portfolio.py`) for the verified numbers. Longer budgets get closer to the
leader (the winner used ~100k generations).